# 04 - Demand Forecasting (Prophet)
Chronological hold-out evaluation and 60-day forecast for overall demand, using the same functions as the pipeline (`src/ml/demand_forecasting.py`).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
import plotly.express as px
from config.database import read_sql
pd.set_option('display.max_columns', 50)
# NOTE: results reflect whatever data the pipeline loaded (DATA_SOURCE=kaggle -> real Olist, sample -> SYNTHETIC).

In [ ]:
from config.settings import get_settings
from src.ml import demand_forecasting as f
from src.utils.helpers import effective_data_end
cfg = get_settings().forecast
from config.database import get_engine
items = f.load_item_history(get_engine())
daily = f.to_daily_series(items, start=cfg.train_start)
end = effective_data_end(daily.set_index('ds')['y'])
series = daily[daily.ds <= end]
print('forecast origin:', end.date(), '| days:', len(series))

In [ ]:
q = f.check_series_quality(series, cfg)
q, 'weekly seasonality supported:', f.weekly_seasonality_supported(series)

In [ ]:
res = f.forecast_entity(series, cfg)
pd.Series(res['evaluation']['metrics'])

In [ ]:
ho = res['evaluation']['holdout_frame']; fc = res['forecast']
fig = px.line(series.tail(150), x='ds', y='y', title='Actual vs hold-out prediction vs forecast')
fig.add_scatter(x=ho.ds, y=ho.yhat, name='hold-out prediction')
fig.add_scatter(x=fc.ds, y=fc.yhat, name='forecast')
fig.add_scatter(x=fc.ds, y=fc.yhat_upper, name='upper', line=dict(dash='dot'))
fig.add_scatter(x=fc.ds, y=fc.yhat_lower, name='lower', line=dict(dash='dot'))
fig

In [ ]:
pd.DataFrame({h: [fc[fc.horizon_day <= h].yhat.sum()] for h in (7, 30, 60)}, index=['forecast units'])

In [ ]:
read_sql('SELECT model_level, COUNT(*) models, ROUND(AVG(mae),3) mae, ROUND(AVG(rmse),3) rmse, ROUND(AVG(wape),1) wape, ROUND(AVG(naive_mae),3) naive_mae FROM analytics.forecast_accuracy GROUP BY 1')